
# STEP 35 — Publication Table 3 Generator

เปรียบเทียบ **Labelled cohort (n=212)** กับ **Unlabelled cohort (n=189)** จากข้อมูลจริง

Input:
- `34_Table_Generator_Input/evaluation_summary_with_labels.csv`
- `34_Table_Generator_Input/oasis_cross-sectional*.xlsx`

Output:
- `35_Table3_Output/Table3_Labelled_vs_Unlabelled.xlsx`
- `35_Table3_Output/Table3_Labelled_vs_Unlabelled.csv`
- `35_Table3_Output/Table3_Publication.docx`
- `35_Table3_Output/Table3_Audit_Report.xlsx`
- `35_Table3_Output/Table3_Manifest.json`

> ใช้ `Kernel → Restart & Run All`


In [ ]:

from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

ROOTS = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd(),
]
ROOT = next((p for p in ROOTS if p.exists()), Path.cwd())
INPUT_DIR = ROOT / "34_Table_Generator_Input"
OUTPUT_DIR = ROOT / "35_Table3_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_FILE = INPUT_DIR / "evaluation_summary_with_labels.csv"
CLINICAL_FILES = sorted(
    list(INPUT_DIR.glob("oasis_cross-sectional*.xlsx"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.xls"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.csv"))
)
DETAIL_JSON = ROOT / "evaluation_case_details.json"

if not PRED_FILE.exists():
    raise FileNotFoundError(PRED_FILE)
if not CLINICAL_FILES:
    raise FileNotFoundError("No OASIS clinical spreadsheet found")

CLINICAL_FILE = CLINICAL_FILES[0]

print("Prediction:", PRED_FILE)
print("Clinical  :", CLINICAL_FILE)
print("Output    :", OUTPUT_DIR)


In [ ]:

def load_table(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path)

def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    return None

def normalize_subject_id(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().upper().replace("-", "_").replace(" ", "_")
    t = re.sub(r"_+", "_", t)
    m = re.search(r"(OAS1_\d{4}_MR\d+)", t)
    return m.group(1) if m else t

def normalize_label(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().upper()
    if t in {"CN","0","CONTROL","NORMAL","COGNITIVELY NORMAL"}: return "CN"
    if t in {"AD","1","ALZHEIMER","ALZHEIMER'S DISEASE","DEMENTIA","DEMENTED"}: return "AD"
    return np.nan

def sex_norm(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().upper()
    return {"F":"F","FEMALE":"F","M":"M","MALE":"M"}.get(t, np.nan)

def fmt_mean_sd(s, digits=1):
    x = pd.to_numeric(s, errors="coerce").dropna()
    return "NA" if len(x)==0 else f"{x.mean():.{digits}f} ± {x.std(ddof=1):.{digits}f}"

def fmt_median_iqr(s, digits=0):
    x = pd.to_numeric(s, errors="coerce").dropna()
    if len(x)==0: return "NA"
    q1, med, q3 = np.percentile(x,[25,50,75])
    return f"{med:.{digits}f} ({q1:.{digits}f}–{q3:.{digits}f})"

def fmt_n_pct(n,total):
    return "0 (0.0%)" if total==0 else f"{int(n)} ({100*n/total:.1f}%)"

def fmt_p(p):
    if p is None or not np.isfinite(p): return "—"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def welch_p(a,b):
    a = pd.to_numeric(a,errors="coerce").dropna()
    b = pd.to_numeric(b,errors="coerce").dropna()
    return np.nan if len(a)<2 or len(b)<2 else float(stats.ttest_ind(a,b,equal_var=False).pvalue)

def mannwhitney_p(a,b):
    a = pd.to_numeric(a,errors="coerce").dropna()
    b = pd.to_numeric(b,errors="coerce").dropna()
    return np.nan if len(a)==0 or len(b)==0 else float(stats.mannwhitneyu(a,b,alternative="two-sided").pvalue)

def chi_or_fisher(table):
    arr = np.asarray(table,dtype=int)
    if arr.shape != (2,2) or arr.sum()==0: return np.nan,"NA"
    chi2,p,dof,expected = stats.chi2_contingency(arr)
    if (expected < 5).any():
        _,p = stats.fisher_exact(arr)
        return float(p),"Fisher exact"
    return float(p),"Chi-square"


In [ ]:

pred = load_table(PRED_FILE)
clinical = load_table(CLINICAL_FILE)

id_col = first_existing(pred.columns, ["case_id","subject_key","subject_id","id"])
label_col = first_existing(pred.columns, ["ground_truth","ground_truth_original","clinical_label"])
status_col = first_existing(pred.columns, ["readiness_status","status"])
readiness_success_col = first_existing(pred.columns, ["readiness_success"])
failure_step_col = first_existing(pred.columns, ["failure_step"])
error_message_col = first_existing(pred.columns, ["error_message"])

if id_col is None or label_col is None:
    raise KeyError(f"Missing required columns: id={id_col}, label={label_col}")

pred["_subject_id"] = pred[id_col].map(normalize_subject_id)
pred["_label"] = pred[label_col].map(normalize_label)
pred["_cohort"] = np.where(pred["_label"].notna(),"Labelled","Unlabelled")
pred = pred.drop_duplicates("_subject_id",keep="first").copy()

print(pred["_cohort"].value_counts())

if len(pred)!=401 or (pred["_cohort"]=="Labelled").sum()!=212 or (pred["_cohort"]=="Unlabelled").sum()!=189:
    raise ValueError("Cohort counts do not match 401 / 212 / 189")


In [ ]:

clinical_id_col = first_existing(clinical.columns, ["ID","subject_id","subject","case_id"])
if clinical_id_col is None:
    raise KeyError("Clinical ID column not found")

clinical["_subject_id"] = clinical[clinical_id_col].map(normalize_subject_id)
clinical_unique = clinical.drop_duplicates("_subject_id",keep="first").copy()

merged = pred.merge(
    clinical_unique,
    on="_subject_id",
    how="left",
    suffixes=("_pred","_clinical"),
    indicator=True
)

age_col = first_existing(merged.columns, ["Age","age_years"])
sex_col = first_existing(merged.columns, ["M/F","sex","gender"])
cdr_col = first_existing(merged.columns, ["CDR","cdr_score"])

labelled = merged[merged["_cohort"]=="Labelled"].copy()
unlabelled = merged[merged["_cohort"]=="Unlabelled"].copy()

print("Clinical matches:", int((merged["_merge"]=="both").sum()), "/", len(merged))
print("Age:", age_col, "| Sex:", sex_col, "| CDR:", cdr_col)


In [ ]:

# Readiness-warning definition from explicit pipeline fields only.
def nonempty(series):
    s = series.fillna("").astype(str).str.strip().str.upper()
    return ~s.isin(["","NONE","NAN","NULL"])

warn = pd.Series(False,index=merged.index)

if status_col is not None:
    warn |= merged[status_col].fillna("").astype(str).str.upper().str.contains("WARN")
if readiness_success_col is not None:
    warn |= merged[readiness_success_col].astype(str).str.upper().isin(["FALSE","0"])
if failure_step_col is not None:
    warn |= nonempty(merged[failure_step_col])
if error_message_col is not None:
    warn |= nonempty(merged[error_message_col])

merged["_warning"] = warn
labelled = merged[merged["_cohort"]=="Labelled"].copy()
unlabelled = merged[merged["_cohort"]=="Unlabelled"].copy()

print("Warnings labelled  :", int(labelled["_warning"].sum()))
print("Warnings unlabelled:", int(unlabelled["_warning"].sum()))


In [ ]:

# Accept only a real, explicit slice-count column.
SLICE_CANDIDATES = [
    "slice_count","n_slices","num_slices","number_of_slices",
    "slices_analysed","slices_analyzed","selected_slice_count",
    "usable_slice_count","analyzed_slice_count","analysed_slice_count"
]
slice_col = first_existing(merged.columns, SLICE_CANDIDATES)
print("Slice-count column:", slice_col)

# NOTE:
# selected_volume_index and selected_volume_score are intentionally excluded.


In [ ]:

rows = []

# Age
if age_col is not None:
    rows.append([
        "Age (years), mean ± SD",
        fmt_mean_sd(labelled[age_col],1),
        fmt_mean_sd(unlabelled[age_col],1),
        fmt_p(welch_p(labelled[age_col],unlabelled[age_col]))
    ])
else:
    rows.append(["Age (years), mean ± SD","NA","NA","—"])

# Sex
if sex_col is not None:
    L = labelled[sex_col].map(sex_norm)
    U = unlabelled[sex_col].map(sex_norm)
    lf,lm = int((L=="F").sum()),int((L=="M").sum())
    uf,um = int((U=="F").sum()),int((U=="M").sum())
    p_sex, sex_test = chi_or_fisher([[lf,lm],[uf,um]])
    rows.append([
        "Sex, female / male, n (%)",
        f"{fmt_n_pct(lf,len(labelled))} / {fmt_n_pct(lm,len(labelled))}",
        f"{fmt_n_pct(uf,len(unlabelled))} / {fmt_n_pct(um,len(unlabelled))}",
        fmt_p(p_sex)
    ])
else:
    sex_test = "NA"
    rows.append(["Sex, female / male, n (%)","NA","NA","—"])

# Slice count
if slice_col is not None:
    rows.append([
        "Slices analysed per subject, median (IQR)",
        fmt_median_iqr(labelled[slice_col],0),
        fmt_median_iqr(unlabelled[slice_col],0),
        fmt_p(mannwhitney_p(labelled[slice_col],unlabelled[slice_col]))
    ])
else:
    rows.append([
        "Slices analysed per subject, median (IQR)",
        "Not available","Not available","—"
    ])

# Warnings
lw,uw = int(labelled["_warning"].sum()),int(unlabelled["_warning"].sum())
p_warn, warn_test = chi_or_fisher([[lw,len(labelled)-lw],[uw,len(unlabelled)-uw]])
rows.append([
    "Readiness-assessment warnings, n (%)",
    fmt_n_pct(lw,len(labelled)),
    fmt_n_pct(uw,len(unlabelled)),
    fmt_p(p_warn)
])

# Reason absent
rows.append([
    "Reason for absent label",
    "Not applicable",
    "No usable CDR-based ground-truth label in the linked clinical record",
    "—"
])

table3 = pd.DataFrame(rows,columns=[
    "Characteristic","Labelled (n = 212)","Unlabelled (n = 189)","p-value"
])
display(table3)


In [ ]:

# Audit reports
cohort_audit = pd.DataFrame({
    "Check":["Processed","Labelled","Unlabelled","Clinical matches"],
    "Value":[len(merged),len(labelled),len(unlabelled),int((merged["_merge"]=="both").sum())],
    "Expected":[401,212,189,"recorded"]
})

missing_report = pd.DataFrame([
    {
        "Variable":"Age",
        "Labelled missing": int(labelled[age_col].isna().sum()) if age_col else len(labelled),
        "Unlabelled missing": int(unlabelled[age_col].isna().sum()) if age_col else len(unlabelled)
    },
    {
        "Variable":"Sex",
        "Labelled missing": int(labelled[sex_col].isna().sum()) if sex_col else len(labelled),
        "Unlabelled missing": int(unlabelled[sex_col].isna().sum()) if sex_col else len(unlabelled)
    },
    {
        "Variable":"Slice count",
        "Labelled missing": int(labelled[slice_col].isna().sum()) if slice_col else len(labelled),
        "Unlabelled missing": int(unlabelled[slice_col].isna().sum()) if slice_col else len(unlabelled)
    }
])

display(cohort_audit)
display(missing_report)


In [ ]:

footnote = (
    "Values compare the 212 subjects with usable diagnostic labels and the 189 "
    "processed examinations without a usable label. Age was compared using Welch's "
    "independent-samples t-test; sex and readiness-warning frequencies were compared "
    "using categorical tests as appropriate. "
)
if slice_col is not None:
    footnote += "Slice counts were compared using the Mann–Whitney U test. "
else:
    footnote += (
        "An explicit analysed-slice-count variable was not available in the current "
        "pipeline records; no surrogate variable was substituted. "
    )
footnote += "Missing clinical metadata were not imputed."

print(footnote)


In [ ]:

xlsx_path = OUTPUT_DIR/"Table3_Labelled_vs_Unlabelled.xlsx"
csv_path = OUTPUT_DIR/"Table3_Labelled_vs_Unlabelled.csv"
audit_path = OUTPUT_DIR/"Table3_Audit_Report.xlsx"

table3.to_csv(csv_path,index=False,encoding="utf-8-sig")

with pd.ExcelWriter(xlsx_path,engine="openpyxl") as writer:
    table3.to_excel(writer,sheet_name="Table 3",index=False,startrow=2)
    ws = writer.book["Table 3"]
    ws["A1"] = (
        "Table 3. Comparison between the 212 subjects included in the quantitative "
        "evaluation and the 189 processed examinations without a usable clinical label."
    )
    foot_row = len(table3)+5
    ws.cell(row=foot_row,column=1,value=footnote)
    ws.merge_cells(start_row=foot_row,start_column=1,end_row=foot_row+3,end_column=4)

    from openpyxl.styles import Font,Alignment
    ws["A1"].font = Font(bold=True,size=11)
    for c in ws[3]:
        c.font = Font(bold=True)
        c.alignment = Alignment(horizontal="center")
    for row in ws.iter_rows(min_row=4,max_row=3+len(table3),min_col=1,max_col=4):
        for c in row:
            c.alignment = Alignment(wrap_text=True,vertical="center")
    for col,width in {"A":42,"B":30,"C":46,"D":14}.items():
        ws.column_dimensions[col].width = width
    ws.cell(row=foot_row,column=1).alignment = Alignment(wrap_text=True,vertical="top")

with pd.ExcelWriter(audit_path,engine="openpyxl") as writer:
    cohort_audit.to_excel(writer,sheet_name="Cohort Audit",index=False)
    missing_report.to_excel(writer,sheet_name="Missing Data",index=False)
    merged.to_excel(writer,sheet_name="Merged 401",index=False)

print("Saved:",xlsx_path)
print("Saved:",csv_path)
print("Saved:",audit_path)


In [ ]:

# Optional Word
try:
    from docx import Document
    from docx.shared import Pt
    doc = Document()
    p = doc.add_paragraph()
    r = p.add_run(
        "Table 3. Comparison between the 212 subjects included in the quantitative "
        "evaluation and the 189 processed examinations without a usable clinical label."
    )
    r.bold = True
    r.font.size = Pt(10)
    t = doc.add_table(rows=1,cols=4)
    t.style = "Table Grid"
    for j,c in enumerate(table3.columns):
        t.rows[0].cells[j].text = c
    for _,row in table3.iterrows():
        cells = t.add_row().cells
        for j,val in enumerate(row):
            cells[j].text = str(val)
    fp = doc.add_paragraph()
    fr = fp.add_run(footnote)
    fr.italic = True
    fr.font.size = Pt(8)
    doc.save(OUTPUT_DIR/"Table3_Publication.docx")
    print("Saved Word table")
except Exception as e:
    print("Word export skipped:",e)


In [ ]:

manifest = {
    "processed": len(merged),
    "labelled": len(labelled),
    "unlabelled": len(unlabelled),
    "id_column": id_col,
    "label_column": label_col,
    "clinical_id_column": clinical_id_col,
    "age_column": age_col,
    "sex_column": sex_col,
    "slice_count_column": slice_col,
    "warnings_labelled": int(labelled["_warning"].sum()),
    "warnings_unlabelled": int(unlabelled["_warning"].sum()),
    "table3": table3.to_dict(orient="records"),
    "footnote": footnote,
}
with open(OUTPUT_DIR/"Table3_Manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2)

print("STEP 35 COMPLETE")
